# Лабораторная работа № 2. Векторные представления и память переводов

**Дисциплина:** Основы машинного обучения
**Направление:** Цифровая лингвистика и локализация

## Цель

Разобраться, как устроен нечёткий поиск по памяти переводов — та самая
подсказка «совпадение 87%», которую показывают Trados, memoQ и Smartcat, —
и понять, чем поверхностное сходство строк отличается от векторного.

## Задачи

1. Реализовать расстояние Левенштейна и получить из него процент совпадения.
2. Представить сегменты векторами (TF-IDF) и измерить косинусную близость.
3. **Главное:** найти сегменты, на которых две меры расходятся, и объяснить,
   почему.
4. Попробовать кластеризацию и понять, что она может и чего не может.
5. Извлечь кандидатов в термины и построить карту корпуса.

## Что важно понять

Память переводов — это поиск ближайшего соседа. Весь вопрос в том,
**что считать расстоянием между двумя текстами**. Разные ответы дают разные
подсказки переводчику, и ошибаются они по-разному.

> ### Интерактивная версия этой работы
>
> [Playground «Память переводов»](https://konkin-nikita.ru/gd_learn/tm/) ищет подсказку по одному сегменту
> сразу четырьмя мерами и кладёт все 160 сегментов на один график «Левенштейн ×
> косинус» — там сразу видно, где меры расходятся. Это главное задание работы
> (задание 3).
>
> Вкладка «Кластеризация» показывает ARI на трёх представлениях вместе с
> разбросом по восьми запускам: разброс шире разницы между представлениями.

---
## 1. Память переводов

Память переводов (translation memory, TM) — база пар «исходный сегмент →
перевод». Когда в новом файле встречается сегмент, система ищет в TM похожий
и предлагает готовый перевод. Наш корпус — маленькая TM: `en` → `ru_ref`.

**В какую сторону ищем.** Классический сценарий CAT — поиск по стороне
оригинала: пришла новая английская строка, ищем похожую английскую в памяти,
предлагаем её русский перевод. Мы будем работать со стороной перевода:
пришла новая **русская** строка, ищем похожие уже переведённые русские.
Это тоже реальная задача — **контроль консистентности**: проследить, чтобы
одно и то же не оказалось переведено в проекте двумя разными способами.
Такой выбор позволяет заодно увидеть, как на поиск влияет русская морфология.
Поиск по английской стороне вынесен в задание 5.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_colwidth", 70)

tm = pd.read_csv("data/loc_corpus.csv")[["id", "type", "en", "ru_ref"]]
print("Сегментов в памяти переводов:", len(tm))
tm.head(4)

### Новые сегменты

В новой версии продукта строки меняются: где-то поправили число, где-то
заменили слово, где-то переписали фразу целиком. Именно такие сегменты
приходят на перевод.

In [ ]:
QUERIES = [
    "Сохранить все изменения",                       # добавлено слово
    "Удалить папку «{name}»?",                       # заменён термин
    "Пароль должен содержать не менее 12 символов",   # изменено число
    "Попробуйте бесплатно 14 дней. Карта не нужна.",  # изменено число
    "Резервные копии создаются каждое воскресенье в 03:00 по времени сервера.",
    "Изменения вступают в силу после перезапуска службы.",
    "Не удалось подключиться к базе данных",         # заменена концовка
    "Настоящие Правила регулируют использование Платформы.",  # заменены термины
    "Настройки поиска",                              # переставлены слова
    "Сохранение изменений",                          # номинализация
    "Пожалуйста, очистите кэш и повторите попытку",  # короткий сегмент внутри длинного
    "По вашему запросу результатов нет",             # синонимия без общих слов
    "Максимальный размер вложения — 50 МБ.",         # похожего сегмента нет
]
for q in QUERIES:
    print("•", q)

---
## 2. Мера первая: расстояние Левенштейна

**Расстояние Левенштейна** между двумя строками — минимальное число
односимвольных операций (вставка, удаление, замена), которыми одну строку
можно превратить в другую. Именно оно в разных вариациях лежит в основе
процента совпадения в CAT-инструментах.

Реализуем его сами: алгоритм короткий, и полезно увидеть, что внутри.

In [ ]:
def levenshtein(a, b):
    '''Минимальное число вставок, удалений и замен, превращающих a в b.'''
    prev_row = list(range(len(b) + 1))
    for i, ch_a in enumerate(a, start=1):
        cur_row = [i]
        for j, ch_b in enumerate(b, start=1):
            insertion    = cur_row[j - 1] + 1
            deletion     = prev_row[j] + 1
            substitution = prev_row[j - 1] + (ch_a != ch_b)
            cur_row.append(min(insertion, deletion, substitution))
        prev_row = cur_row
    return prev_row[-1]

print(levenshtein("кот", "кит"))            # 1 замена
print(levenshtein("кот", "скот"))           # 1 вставка
print(levenshtein("Сохранить", "Сохранять"))

In [ ]:
def match_percent(a, b):
    '''Процент совпадения в духе CAT-инструментов: 100% — строки идентичны.'''
    if not a and not b:
        return 100.0
    return (1 - levenshtein(a, b) / max(len(a), len(b))) * 100

print(f"{match_percent('Сохранить изменения', 'Сохранить все изменения'):.0f}%")
print(f"{match_percent('Сохранить изменения', 'Отменить изменения'):.0f}%")
print(f"{match_percent('Сохранить изменения', 'Соединение восстановлено'):.0f}%")

> **Порог 75%.** В индустрии совпадения ниже примерно 75% обычно не
> показывают: править чужой неподходящий перевод дольше, чем перевести
> заново. Этот порог — не свойство алгоритма, а решение, принятое человеком
> исходя из стоимости ошибки.

In [ ]:
def find_by_levenshtein(query, tm, top_n=3):
    '''Ближайшие сегменты памяти переводов по проценту совпадения.'''
    scores = tm["ru_ref"].apply(lambda s: match_percent(query, s))
    best   = scores.nlargest(top_n)
    return pd.DataFrame({
        "совпадение, %": best.round(1),
        "сегмент из TM": tm.loc[best.index, "ru_ref"],
    })

for q in QUERIES[:3]:
    print("ЗАПРОС:", q)
    print(find_by_levenshtein(q, tm).to_string(index=False))
    print()

---
## 3. Мера вторая: векторное представление и косинус

Превратим сегменты в векторы — как в Л.р. № 1 — и будем измерять угол
между ними.

**Косинусная близость** двух векторов — косинус угла между ними: 1, если
направления совпадают, 0, если векторы ортогональны. Для текстов это
означает «доля общих признаков с учётом их веса».

Важное отличие от Левенштейна: косинус не знает про порядок. Перестановка
слов почти не меняет его, а для Левенштейна это десятки операций.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Символьные n-граммы: как и в Л.р. № 1, для русского они работают лучше
# словарных, потому что устойчивы к словоизменению.
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))
V_tm = vectorizer.fit_transform(tm["ru_ref"])

print("Матрица памяти переводов:", V_tm.shape)
print(f"→ {V_tm.shape[0]} сегментов, каждый описан {V_tm.shape[1]} признаками")

In [ ]:
def find_by_cosine(query, top_n=3):
    '''Ближайшие сегменты TM по косинусной близости векторов.'''
    v = vectorizer.transform([query])
    similarity = cosine_similarity(v, V_tm)[0]
    indices = np.argsort(similarity)[::-1][:top_n]
    return pd.DataFrame({
        "близость": similarity[indices].round(3),
        "сегмент из TM": tm["ru_ref"].iloc[indices].values,
    })

for q in QUERIES[:3]:
    print("ЗАПРОС:", q)
    print(find_by_cosine(q).to_string(index=False))
    print()

---
## 4. Где две меры расходятся

Центральная часть работы. Сравним для каждого запроса лучший ответ
Левенштейна и лучший ответ косинуса.

In [ ]:
comparison = []
for q in QUERIES:
    lev = find_by_levenshtein(q, tm, top_n=1)
    cos = find_by_cosine(q, top_n=1)
    comparison.append({
        "запрос": q,
        "Левенштейн": lev["сегмент из TM"].iloc[0],
        "%": round(lev["совпадение, %"].iloc[0], 1),
        "косинус": cos["сегмент из TM"].iloc[0],
        "близость": cos["близость"].iloc[0],
        "совпали": lev["сегмент из TM"].iloc[0] == cos["сегмент из TM"].iloc[0],
    })

table = pd.DataFrame(comparison)
print("Меры дали один и тот же ответ в", table["совпали"].sum(),
      "случаях из", len(table))
table[["запрос", "Левенштейн", "%", "косинус", "близость", "совпали"]]

In [ ]:
# Развёрнутый разбор случаев, где меры разошлись
divergences = table[~table["совпали"]]
if len(divergences) == 0:
    print("На этих запросах меры не разошлись — добавьте свои (задание 2).")
for _, r in divergences.iterrows():
    print("=" * 74)
    print("ЗАПРОС     :", r["запрос"])
    print(f"Левенштейн : {r['Левенштейн']}   ({r['%']}%)")
    print(f"Косинус    : {r['косинус']}   ({r['близость']})")

Расхождения здесь трёх разных видов, и каждый стоит разобрать отдельно:

**1. Переставлены слова** — «Настройки поиска» против «Поиск по настройкам».
Для косинуса это почти один и тот же вектор: набор символьных n-грамм
изменился мало. Для Левенштейна перестановка — это переписать половину
строки, поэтому он уходит к совершенно постороннему сегменту.
Мера, не знающая про порядок, здесь выигрывает.

**2. Короткий сегмент внутри длинного запроса** — «Пожалуйста, очистите кэш
и повторите попытку» против «Очистить кэш». Левенштейн делит на длину
большей строки, поэтому короткое совпадение внутри длинного запроса
даёт низкий процент по определению — и он снова промахивается.
Косинус нормирует иначе и находит нужное.

**3. Номинализация** — «Сохранение изменений». Левенштейн предлагает
«Сохранить изменения», косинус — «Изменения сохранены». Обе подсказки
разумны, и выбирать между ними должен человек: они различаются
грамматически, а значит подойдут в разных местах интерфейса.

> **Общий принцип.** Левенштейн считает символы и чувствителен к длине и
> порядку. Косинус нормирует на длину и порядок игнорирует, но легко
> обманывается служебной лексикой: два юридических сегмента похожи уже
> потому, что оба канцелярские.
>
> Ни одна из мер не «правильная». Правильная — та, чьи ошибки дешевле для
> конкретного проекта, и выбирает её человек.

### Случай, на котором видны границы обеих мер

Запрос «По вашему запросу результатов нет» и сегмент TM «Ничего не найдено»
значат одно и то же, но не имеют общих слов и почти не имеют общих
символьных n-грамм.

In [ ]:
q = "По вашему запросу результатов нет"
expected = "Ничего не найдено"

print("ЛЕВЕНШТЕЙН:")
print(find_by_levenshtein(q, tm, top_n=3).to_string(index=False))
print("\nКОСИНУС:")
print(find_by_cosine(q, top_n=3).to_string(index=False))

i_expected = tm.index[tm["ru_ref"] == expected][0]
print(f"\nА нужен был сегмент «{expected}»:")
print(f"  процент совпадения : {match_percent(q, expected):.1f}%")
sim_expected = cosine_similarity(vectorizer.transform([q]), V_tm[i_expected])[0][0]
print(f"  косинусная близость: {sim_expected:.3f}")

Обе меры этот случай не находят, и это закономерно: они работают с
**формой**, а не со **смыслом**. Синонимия им недоступна в принципе.

Чтобы находить такие пары, нужны представления, обученные на больших
корпусах, — они и стоят за современными «умными» подсказками в CAT.
Следующий раздел показывает самый простой шаг в эту сторону.

---
## 5. Латентная семантика: шаг от формы к смыслу

Идея **латентно-семантического анализа** (LSA): матрица «сегменты × признаки»
очень разрежена, но если сжать её до нескольких десятков измерений, близкие
по употреблению признаки склеятся, и единицы, встречающиеся в похожих
контекстах, окажутся рядом.

Технически это **снижение размерности** из лекции (слайды 14-15), здесь —
методом усечённого сингулярного разложения (`TruncatedSVD`).

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer

lsa = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)),
    TruncatedSVD(n_components=60, random_state=0),
    Normalizer(copy=False),
)
L_tm = lsa.fit_transform(tm["ru_ref"])

explained = lsa.named_steps["truncatedsvd"].explained_variance_ratio_.sum()
print("После сжатия:", L_tm.shape)
print(f"60 измерений сохраняют {explained:.1%} исходной изменчивости")

In [ ]:
def find_by_lsa(query, top_n=3):
    v = lsa.transform([query])
    similarity = cosine_similarity(v, L_tm)[0]
    indices = np.argsort(similarity)[::-1][:top_n]
    return pd.DataFrame({
        "близость": similarity[indices].round(3),
        "сегмент из TM": tm["ru_ref"].iloc[indices].values,
    })

print("ЗАПРОС:", q)
print(find_by_lsa(q).to_string(index=False))
sim_lsa = cosine_similarity(lsa.transform([q]), L_tm[i_expected:i_expected + 1])[0][0]
print(f"\nблизость до «{expected}»: {sim_lsa:.3f}")

> **Честный вывод.** На 160 коротких сегментах LSA почти ничего не добавляет:
> чтобы «склеить» синонимы, метод должен много раз увидеть их в похожих
> контекстах, а у нас каждая единица встречается один-два раза.
>
> Это тот же урок, что дала кривая обучения в Л.р. № 1: **метод работает не
> сам по себе, а на данных нужного объёма.** Настоящие семантические
> представления обучают на миллиардах слов — и именно поэтому их берут
> готовыми, а не считают на своём проекте.

---
## 6. Кластеризация: обучение без учителя

До сих пор мы либо знали правильные ответы (Л.р. № 1), либо сравнивали пары.
Кластеризация ищет группы, не зная разметки вообще, — это **обучение без
учителя** (лекция, слайды 12-13).

Проверим прямо: найдёт ли `KMeans` наши четыре типа контента, если ему их
не показывать? Мера согласия — **скорректированный индекс Ранда** (ARI):
1.0 — полное совпадение с разметкой, 0.0 — совпадение на уровне случайного.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

clusters = KMeans(n_clusters=4, random_state=0, n_init=10).fit_predict(V_tm)

ari = adjusted_rand_score(tm["type"], clusters)
print(f"ARI = {ari:.3f}   (1.0 — идеально, 0.0 — как случайное разбиение)")
pd.crosstab(clusters, tm["type"])

> **Результат отрицательный, и его нужно понять.** ARI около нуля означает:
> найденные кластеры не имеют отношения к типу контента. Модель не сломалась —
> она честно нашла группы, но **не те, которые интересны нам**.
>
> Причина принципиальная: у кластеризации нет цели «раздели по типу контента».
> Её цель — минимизировать разброс внутри групп. На наших данных
> доминирующий источник разброса — длина сегмента и доля служебной лексики,
> а не регистр. Сравните с Л.р. № 1: там та же векторизация с той же моделью
> давала 0.75, но модель **видела правильные ответы**.
>
> Практический вывод: обучение без учителя не заменяет разметку.
> Оно годится, когда вы готовы принять любую группировку, которую найдут
> данные, — например, чтобы посмотреть, из чего вообще состоит незнакомый
> корпус. Когда группы заданы заранее, нужна разметка.

---
## 7. Извлечение кандидатов в термины

Разметка у нас есть — воспользуемся ею. Термин, характерный для типа
контента, — это единица, которая внутри класса весит заметно больше, чем
за его пределами. Посчитаем эту разницу напрямую.

In [ ]:
word_vec = TfidfVectorizer(min_df=2)      # единица, встретившаяся один раз,
M_words = word_vec.fit_transform(tm["ru_ref"]).toarray()   # ещё не кандидат
words = np.array(word_vec.get_feature_names_out())
print("Словарь кандидатов:", len(words), "единиц")

for ctype in tm["type"].unique():
    mask = (tm["type"] == ctype).values
    weight = M_words[mask].mean(axis=0) - M_words[~mask].mean(axis=0)
    top = words[np.argsort(weight)[-12:]][::-1]
    print(f"\n{ctype.upper()}")
    print("   ", ", ".join(top))

> **Кандидаты — именно кандидаты.** В списках заведомо есть служебная
> лексика: предлоги и союзы, частотные в одном регистре и редкие в другом.
> Отбор настоящих терминов — работа лингвиста, и полностью её не
> автоматизировать. Машина сокращает список с тысяч до десятков;
> решение принимает человек. Долю пригодных кандидатов вы измерите
> в задании 7 — это и есть численная оценка пользы от автоматизации.

---
## 8. Средство: готовые многоязычные эмбеддинги

Два раздела подряд закончились отрицательным результатом: LSA не нашла
синонимию (раздел 5), кластеризация не воспроизвела разметку (раздел 6).
Причина в обоих случаях одна — **представление построено только по нашему
корпусу**, а в нём каждая единица встречается один-два раза.

Средство, которое это лечит, существует и берётся готовым: векторы,
обученные на миллиардах слов. Считать их не нужно — они посчитаны заранее
и лежат в `data/`. Модель и способ расчёта описаны в `data/О_данных.md`.

In [ ]:
import json

E_tm = np.load("data/emb_ru_ref.npy")        # по вектору на сегмент памяти
E_q  = np.load("data/emb_запросы.npy")       # по вектору на запрос
QUERIES_FILE = json.load(open("data/emb_запросы.json", encoding="utf-8"))

print("эмбеддинги памяти:  ", E_tm.shape)
print("эмбеддинги запросов:", E_q.shape)
print("размерность вектора:", E_tm.shape[1], "— вместо", V_tm.shape[1], "у TF-IDF")

Обратите внимание на размерность: 384 числа вместо нескольких тысяч. Вектор
стал короче и при этом **плотным** — ненулевых значений в нём не единицы,
а все. Он кодирует не «какие n-граммы встретились», а «на что этот текст
похож» с точки зрения модели, прочитавшей очень много текста.

In [ ]:
def find_by_embedding(query_index, top_n=3):
    # Ближайшие сегменты TM по косинусу между готовыми эмбеддингами.
    similarity = cosine_similarity(E_q[query_index:query_index + 1], E_tm)[0]
    indices = np.argsort(similarity)[::-1][:top_n]
    return pd.DataFrame({
        "близость": similarity[indices].round(3),
        "сегмент из TM": tm["ru_ref"].iloc[indices].values,
    })

i_q = QUERIES_FILE.index("По вашему запросу результатов нет")
print("ЗАПРОС:", QUERIES_FILE[i_q])

print("\nЛЕВЕНШТЕЙН:")
print(find_by_levenshtein(q, tm, top_n=3).to_string(index=False))
print("\nСИМВОЛЬНЫЕ n-ГРАММЫ:")
print(find_by_cosine(q, top_n=3).to_string(index=False))
print("\nГОТОВЫЕ ЭМБЕДДИНГИ:")
print(find_by_embedding(i_q, top_n=3).to_string(index=False))

sim_emb = cosine_similarity(E_q[i_q:i_q + 1], E_tm)[0]
rank = list(np.argsort(sim_emb)[::-1]).index(i_expected) + 1
print(f"\n«{expected}»: близость {sim_emb[i_expected]:.3f}, ранг {rank} из {len(tm)}")

> **Средство работает — и работает не идеально.** Сегмент «Ничего не найдено»
> поднялся с глубины корпуса на второе место. Ни Левенштейн, ни символьные
> n-граммы его не находили вообще: у запроса и ответа нет ни одного общего
> слова.
>
> Но первое место занял посторонний сегмент. Эмбеддинги ловят **общую тему**,
> а не точное соответствие, и на коротких строках путают «ничего нет» с
> другими отрицательными конструкциями. Пользы это не отменяет: просмотреть
> три подсказки быстрее, чем переводить с нуля.

### То же средство для кластеризации

В разделе 6 кластеризация на символьных n-граммах дала ARI около нуля.
Повторим её на эмбеддингах: представление другое, метод тот же.

In [ ]:
clusters_emb = KMeans(n_clusters=4, random_state=0, n_init=10).fit_predict(E_tm)

ari_table = pd.DataFrame([
    {"представление": "символьные n-граммы (свой корпус)",
     "ARI": round(adjusted_rand_score(tm["type"], clusters), 3)},
    {"представление": "готовые эмбеддинги",
     "ARI": round(adjusted_rand_score(tm["type"], clusters_emb), 3)},
])
print(ari_table.to_string(index=False))
print()
print(pd.crosstab(clusters_emb, tm["type"]).to_string())

> **Тот же вывод в третий раз, и он главный в работе.** ARI заметно вырос —
> но до значения, которое всё ещё далеко от единицы.
>
> Во сколько именно раз, зависит от машины: у символьных n-грамм `KMeans`
> приходит к разным локальным минимумам при разной реализации BLAS, и
> разброс по зёрнам там шире разницы между представлениями. У эмбеддингов
> разбиение устойчиво. Сравнивайте порядок величины, а не третий знак.
>
> Смена представления **сдвинула** результат, но задачу не решила, потому
> что задача поставлена неверно: кластеризация ищет группы, которые есть в
> данных, а нам нужны группы, которые придумал человек. Совпадать они не
> обязаны ни при каком представлении.
>
> **Средство улучшает то, на что оно рассчитано.** Готовые эмбеддинги лечат
> нехватку данных для построения представления — и потому помогают поиску
> по смыслу. Они не лечат несовпадение цели, и потому кластеризация
> остаётся плохой заменой разметке.

### Цена средства

| | символьные n-граммы | готовые эмбеддинги |
|---|---|---|
| откуда берутся | считаются на своём корпусе за секунду | обучены заранее на миллиардах слов |
| зависимости | только scikit-learn | `torch`, `transformers`, около 500 МБ |
| работа офлайн | да | да, если веса скачаны заранее |
| понятность признаков | видно, какая n-грамма сработала | 384 числа без интерпретации |
| синонимия | не ловится | ловится частично |

Эмбеддинги посчитаны заранее и лежат в `data/` именно поэтому: чтобы работа
выполнялась без установки тяжёлых библиотек и без сети. В реальном проекте
этот расчёт — отдельный этап конвейера, а не строка в ноутбуке.

---
## 9. Карта корпуса

Векторы у нас 60-мерные, а посмотреть на них хочется на плоскости.
Сожмём до двух измерений и нарисуем — то же снижение размерности,
но теперь ради визуализации.

In [ ]:
from sklearn.decomposition import PCA

coords = PCA(n_components=2, random_state=0).fit_transform(L_tm)

fig, ax = plt.subplots(figsize=(7.5, 6))
colors = {"интерфейс": "#4C72B0", "документация": "#55A868",
          "маркетинг": "#C44E52", "юридический": "#8172B2"}
for ctype, color in colors.items():
    mask = (tm["type"] == ctype).values
    ax.scatter(coords[mask, 0], coords[mask, 1],
               c=color, label=ctype, s=45, alpha=0.75,
               edgecolors="white", linewidths=0.6)
ax.set_xlabel("компонента 1"); ax.set_ylabel("компонента 2")
ax.set_title("Карта корпуса: сегменты в пространстве признаков")
ax.legend(title="тип контента")
plt.tight_layout()
plt.show()

> **Осторожно с интерпретацией.** Две компоненты сохраняют лишь часть
> изменчивости; перекрытие точек на плоскости не означает, что они
> неразличимы в исходном пространстве. Карта — инструмент для гипотез,
> а не доказательство. Сопоставьте её с результатами разделов 6 и 8: видно ли
> на карте те четыре группы, которых не нашёл `KMeans`?

---
## 10. Собираем подсказчика

Соберём получившееся в одну функцию — упрощённый аналог панели подсказок
в CAT-инструменте.

In [ ]:
def suggest(query, threshold=75.0):
    '''Ищет в памяти переводов сегмент, пригодный для повторного использования.'''
    scores = tm["ru_ref"].apply(lambda s: match_percent(query, s))
    i = scores.idxmax()
    percent = scores.loc[i]
    if percent >= threshold:
        status = ("точное совпадение" if percent == 100
                  else f"нечёткое совпадение {percent:.0f}%")
        return (f"{status}\n"
                f"  найдено в памяти : {tm.loc[i, 'ru_ref']}\n"
                f"  оригинал         : {tm.loc[i, 'en']}\n"
                f"  тип контента     : {tm.loc[i, 'type']}")
    return f"совпадений выше {threshold:.0f}% нет — переводить с нуля (лучшее: {percent:.0f}%)"

for q_ in QUERIES:
    print("─" * 74)
    print("ЗАПРОС:", q_)
    print(suggest(q_))

---
## Задание

1. **Изучите ноутбук** и выполните его сверху вниз без ошибок.

2. **Добавьте пять своих запросов** к списку `QUERIES`. Составьте их так,
   чтобы среди них были: почти точное совпадение, перестановка слов,
   замена термина и синонимичная перефразировка.

3. **Главное задание.** Для каждого своего запроса сравните ответы
   Левенштейна и косинуса. Найдите минимум два случая расхождения и для
   каждого объясните, какая мера дала более полезный переводчику ответ и
   почему. Объяснение должно опираться на устройство меры, а не на впечатление.

4. **Исследуйте порог.** В функции `suggest` меняйте `threshold` от 50 до 95.
   При каком значении система начинает предлагать бесполезные подсказки?
   Сколько полезных подсказок теряется при пороге 90? Оформите таблицей.

5. **Поиск по стороне оригинала.** Повторите разделы 2-4 для английской
   стороны: память — столбец `en`, запросы — английские варианты ваших
   сегментов. Сравните с русской стороной по двум вопросам:
   (а) даёт ли Левенштейн на английском более высокий процент совпадения
   при той же по смыслу правке; (б) насколько проседает поиск по русской
   стороне при переходе с символьных n-грамм на словарные
   (`TfidfVectorizer()` без параметров) и насколько — по английской.
   Объясните разницу через морфологический тип языков.

6. **Кластеризация.** Измените число кластеров на 3, 6 и 8 и каждый раз
   посчитайте ARI. Растёт ли согласие с разметкой? Затем попробуйте
   кластеризовать не `V_tm`, а `L_tm` (сжатое представление). Меняется ли
   вывод раздела 6? Ответьте на вопрос: можно ли подбором параметров
   заставить кластеризацию воспроизвести разметку, и что это означает.

7. **Извлечение терминологии.** Из списков раздела 7 отберите вручную те
   единицы, которые вы как лингвист признали бы терминами. Посчитайте долю
   пригодных кандидатов по каждому классу. Где автоматика полезнее и почему?

8. **Проверьте средство из раздела 8 на своих запросах.** Для каждого из пяти
   ваших запросов (задание 2) сравните три способа поиска: Левенштейн,
   символьные n-граммы, готовые эмбеддинги. Постройте таблицу «запрос —
   лучший ответ каждого способа». На скольких запросах эмбеддинги дали ответ,
   до которого не добрались первые два? На скольких — испортили верный ответ?

9. **Когда средство не нужно.** Найдите среди своих запросов такой, где
   готовые эмбеддинги проигрывают простому Левенштейну, и объясните почему.
   Сформулируйте правило: при каких свойствах сегмента имеет смысл тратиться
   на эмбеддинги, а при каких достаточно посимвольной меры.

### Требования к отчёту

* ноутбук выполняется сверху вниз на чистом окружении;
* в задании 3 приведены конкретные пары сегментов, а не общие рассуждения;
* отрицательные результаты (разделы 5 и 6) объяснены, а не пропущены;
* в задании 8 приведена таблица по всем пяти запросам;
* вывод содержит ответ: какую меру вы выбрали бы для проекта локализации
  интерфейса и почему.